# CLIP

### 双塔结构

CLIP由两个独立编码器组成：
- 图像 ─→ Image Encoder ─→ 图像向量
- 文本 ─→ Text Encoder  ─→ 文本向量

### 损失函数

- 图像-文本损失：$$\mathcal L_{I\rightarrow T} = -\frac1N \sum_{i=1}^{N} \log \frac{\exp(s_{ii}/\tau)}{\sum_{j=1}^{N}\exp(s_{ij}/\tau)}$$
- 文本-图像损失：$$\mathcal L_{T\rightarrow I} = -\frac1N \sum_{i=1}^{N} \log \frac{\exp(s_{ij}/\tau)}{\sum_{j=1}^{N}\exp(s_{jj}/\tau)}$$
- 总损失：平均图像-文本损失 + 平均文本-图像损失
$$\mathcal L = \frac{\mathcal L_{I\rightarrow T} + \mathcal L_{T\rightarrow I}}{2}$$




In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


def clip_loss(
    image_features: torch.Tensor,
    text_features: torch.Tensor,
    temperature: float = 0.07,
) -> torch.Tensor:
    """
    Args:
        image_features: [B, D] 图像编码器输出，B=batch_size, D=特征维度
        text_features:  [B, D] 文本编码器输出，形状与图像特征一致
        temperature:     温度系数，控制 softmax 分布的平滑程度
                         越小 → 分布越尖锐（对正样本更严格）
                         越大 → 分布越平滑（容忍更多噪声）
                         
    Returns:
        标量损失值，= (image→text 交叉熵 + text→image 交叉熵) / 2
    """

    if image_features.shape != text_features.shape:
        raise ValueError("图像特征和文本特征形状必须一致")

    batch_size = image_features.shape[0]

    # L2归一化
    # 归一化后，两个向量的内积等价于它们的余弦相似度
    image_features = F.normalize(
        image_features,
        dim=-1,
    )
    text_features = F.normalize(
        text_features,
        dim=-1,
    )

    # 构造相似度矩阵：[B, D] @ [D, B] → [B, B]
    # logits[i][j] = 第 i 张图像与第 j 条文本的余弦相似度 / temperature
    # 主对角线 logits[i][i] 是正确配对的得分
    logits = (
        image_features @ text_features.T
    ) / temperature

    # 构造标签：正确图文对位于主对角线位置
    # labels = [0, 1, 2, ..., B-1]，即第 i 行正确标签是第 i 列
    labels = torch.arange(
        batch_size,
        device=logits.device,
    )

    image_to_text_loss = F.cross_entropy(
        logits,
        labels,
    )

    text_to_image_loss = F.cross_entropy(
        logits.T,
        labels,
    )

    return (
        image_to_text_loss
        + text_to_image_loss
    ) / 2

In [ ]:
# 2. 模拟一个简单的"图文编码器"（用线性层代替）

class SimpleEncoder(torch.nn.Module):
    """用两层 MLP 模拟图像/文本编码器，将输入映射到固定维度特征空间"""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
def demo_training():
    # --- 超参数 ---
    batch_size = 16
    input_dim = 64       # 原始输入维度（如像素、词向量等）
    feature_dim = 128    # 公共特征空间维度
    temperature = 0.07
    num_steps = 200

    # --- 构造模拟数据：生成 B 对"语义对齐"的图文输入 ---
    # 每对共享同一个基础向量 + 各自噪声，模拟真实图文对应关系
    torch.manual_seed(42)
    base_vectors = torch.randn(batch_size, input_dim)  # [B, input_dim]

    # 图像编码器和文本编码器
    image_encoder = SimpleEncoder(input_dim, 256, feature_dim)
    text_encoder = SimpleEncoder(input_dim, 256, feature_dim)
    optimizer = torch.optim.Adam(
        list(image_encoder.parameters()) + list(text_encoder.parameters()),
        lr=1e-3,
    )

    loss_history = []

    for step in range(num_steps):
        # 每步生成新的 batch：同一对图文共享 base_vector，但加不同噪声
        image_input = base_vectors + 0.1 * torch.randn(batch_size, input_dim)
        text_input = base_vectors + 0.1 * torch.randn(batch_size, input_dim)

        image_features = image_encoder(image_input)
        text_features = text_encoder(text_input)

        loss = clip_loss(image_features, text_features, temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
    
    plt.plot(loss_history)
    plt.title("CLIP Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.show()

    return image_encoder, text_encoder

demo_training()